# Geração de dataset sintético — v3 (Tech Challenge Fase 3)

Gera o dataset de **fine-tuning** (`dataset_medico.jsonl`) usando a API do Claude.

## Ordem de execução (importante)

O pipeline tem uma ordem que **não deve ser alterada**:

`gerar → deduplicar → normalizar desfecho → limpar meta-vazamento → ANONIMIZAR → verificar → salvar`

A anonimização vem **por último** de propósito. Se você rodar a célula de geração novamente depois de anonimizar, os exemplos novos entram crus — nesse caso, rode toda a seção de limpeza de novo.

## Mudanças da v2 para a v3
- Anonimização reescrita: captura N iniciais (`R.S.`, `R.M.T.S.`) numa passada só, sem deixar resíduo
- Correção automática de espaços e pontuação após substituição
- Normalização do rótulo de desfecho para `DESFECHO: <RÓTULO>` (parsing trivial no LangGraph)
- Limpeza de meta-vazamento ("identificação fictícia") embutida no pipeline
- Lista de termos de guardrail ampliada — a v2 subestimava a cobertura real
- Célula de verificação final que roda todas as checagens de uma vez

## Antes de compartilhar com o grupo
- A API key fica no Colab Secrets, nunca hardcoded
- Rode uma vez, baixe o `.jsonl`, suba no repositório
- O notebook de treino não depende deste nem de API key
- `Editar → Limpar todos os outputs` antes de compartilhar


## 1. Instalação

In [ ]:
!pip install -q anthropic


## 2. Autenticação

Colab Secrets (ícone 🔑 na barra lateral) com o toggle "Acesso ao bloco de notas" **ativado**.

In [ ]:
import anthropic
from google.colab import userdata

client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
print("Cliente autenticado.")


## 3. Configuração

`cenarios_clinicos` tem o maior peso: é a categoria que treina o modelo a emitir o rótulo de desfecho que o seu grafo LangGraph vai usar para rotear.

In [ ]:
CATEGORIAS = {
    "protocolos": "protocolos clínicos internos (sepse, dor torácica, AVC, pré-operatório, TEP, cetoacidose)",
    "faq_medicos": "perguntas frequentes de médicos sobre condutas e procedimentos internos do hospital",
    "laudos": "modelos e estrutura de laudo de exame (imagem, laboratorial, anatomopatológico)",
    "receitas": "modelos de receita e regras de prescrição (incluindo controlados)",
    "cenarios_clinicos": (
        "cenários clínicos em que o médico apresenta dados de um paciente e o assistente precisa decidir "
        "entre: verificar exames pendentes, sugerir conduta baseada em protocolo, ou emitir alerta de gravidade"
    ),
}

META_POR_CATEGORIA = {
    "protocolos": 20,
    "faq_medicos": 20,
    "laudos": 15,
    "receitas": 15,
    "cenarios_clinicos": 25,
}

EXEMPLOS_POR_LOTE = 5
MODELO = "claude-sonnet-4-6"

print("Total planejado:", sum(META_POR_CATEGORIA.values()), "exemplos")


## 4. Prompts

Três ajustes em relação à v2:
- **Regra 3 (guardrail) agora é condicional**: exigida em conduta/prescrição, proibida em perguntas puramente informativas. Um guardrail que aparece em 100% das respostas vira ruído — o modelo aprende reflexo, não discriminação.
- **Regra 6**: proíbe o modelo de escrever "fictício" dentro do conteúdo do laudo/receita (o vazamento de meta-instrução que apareceu na v2).
- **Regra 7**: fixa o formato `DESFECHO: <RÓTULO>` na primeira linha dos cenários clínicos.

In [ ]:
SYSTEM_PROMPT_GERACAO = """Você gera dados sintéticos de treinamento para um assistente clínico de apoio à decisão hospitalar.

REGRAS ABSOLUTAS:
1. Todos os dados são inventados. Nunca use nomes reais, CPFs, datas reais ou qualquer dado identificável.
2. Sempre cite uma fonte de protocolo interno no formato "Protocolo Interno PROT-0XX" ou modelo interno (ex.: "LAUDO-IMG-01", "RX-CTRL-01").
3. GUARDRAIL CONDICIONAL:
   - Se a resposta envolver sugestão de conduta, tratamento, dose ou prescrição, ela DEVE terminar com uma destas frases exatas:
     "Esta orientação é um apoio à decisão e requer validação do médico responsável antes de qualquer conduta."
     "Sugestão baseada em protocolo interno; requer validação humana antes da prescrição."
   - Se a pergunta for puramente informativa/administrativa (estrutura de um laudo, campos obrigatórios de uma receita, fluxo de solicitação de exame), NÃO inclua a ressalva. Ela só faz sentido onde há conduta clínica em jogo.
4. O assistente NUNCA prescreve diretamente. Ele descreve o que o protocolo prevê e encaminha para validação.
5. Respostas concisas: máximo de 5 linhas. Prefira listas numeradas curtas a parágrafos longos.
6. NUNCA escreva as palavras "fictício", "fictícia" ou "sintético" dentro do conteúdo da resposta. Escreva "Identificação do paciente", não "Identificação fictícia do paciente". O caráter sintético do dado é contexto desta tarefa, não parte do documento descrito.
7. Se o exemplo for um cenário clínico, a resposta DEVE começar exatamente com "DESFECHO: " seguido de VERIFICAR_EXAMES, SUGERIR_CONDUTA ou EMITIR_ALERTA, e depois uma quebra de linha."""

INSTRUCAO_CENARIOS = """
Cada exemplo é um cenário em que o médico informa dados de um paciente. Distribua de forma equilibrada entre:
- VERIFICAR_EXAMES: faltam resultados essenciais; o assistente indica quais verificar e recusa conduta definitiva.
- SUGERIR_CONDUTA: há dados suficientes; descreve o que o protocolo prevê, com ressalva de validação humana.
- EMITIR_ALERTA: há sinal de gravidade; sinaliza urgência à equipe.

Campo "input": dados do paciente (idade, sexo, sinais vitais, queixa, exames disponíveis e pendentes).
Campo "instruction": a pergunta do médico.
Campo "output": começa com "DESFECHO: <RÓTULO>", quebra de linha, depois a justificativa numerada.

Para identificar o paciente use apenas idade e sexo (ex.: "Paciente de 58 anos, masculino"). NÃO use iniciais nem nomes.
"""

def prompt_geracao(chave, descricao, quantidade, ja_gerados):
    extra = INSTRUCAO_CENARIOS if chave == "cenarios_clinicos" else ""
    evitar = ""
    if ja_gerados:
        evitar = "\n\nEVITE repetir temas já gerados: " + " | ".join(ja_gerados[-8:])
    return f"""Gere {quantidade} exemplos sintéticos de treinamento para a categoria: "{descricao}".
{extra}
Formato: uma linha por exemplo, JSON válido puro, SEM markdown, SEM crases, SEM texto fora do JSON:
{{"instruction": "...", "input": "", "output": "..."}}

Cada linha deve ser um JSON completo e fechado. Varie a especialidade médica e a complexidade.{evitar}"""

print("Prompts definidos.")


## 5. Geração em lotes

⚠️ Se rodar esta célula mais de uma vez, os exemplos **acumulam** em `todos_exemplos`. Depois de qualquer nova geração, rode todas as células de limpeza (6 a 9) novamente.

In [ ]:
import json as _json
import time

todos_exemplos = []

def parse_jsonl(texto):
    resultados = []
    for linha in texto.strip().splitlines():
        linha = linha.strip()
        if not linha or linha.startswith("```"):
            continue
        try:
            ex = _json.loads(linha)
        except _json.JSONDecodeError:
            continue
        if all(k in ex for k in ("instruction", "input", "output")):
            resultados.append(ex)
    return resultados

for chave, descricao in CATEGORIAS.items():
    meta = META_POR_CATEGORIA[chave]
    gerados_cat, titulos = [], []
    print(f"\n=== {chave} (meta: {meta}) ===")

    tentativas = 0
    while len(gerados_cat) < meta and tentativas < 12:
        tentativas += 1
        faltam = min(EXEMPLOS_POR_LOTE, meta - len(gerados_cat))
        try:
            response = client.messages.create(
                model=MODELO,
                max_tokens=3000,
                system=SYSTEM_PROMPT_GERACAO,
                messages=[{"role": "user", "content": prompt_geracao(chave, descricao, faltam, titulos)}],
            )
        except Exception as e:
            print(f"  erro na API: {e}")
            time.sleep(3)
            continue

        novos = parse_jsonl(response.content[0].text)
        for ex in novos:
            ex["categoria"] = chave
            gerados_cat.append(ex)
            titulos.append(ex["instruction"][:60])
        print(f"  lote {tentativas}: +{len(novos)} (total {len(gerados_cat)}/{meta})")

    todos_exemplos.extend(gerados_cat[:meta])

print(f"\n>>> Total gerado: {len(todos_exemplos)} exemplos")


## 6. Deduplicação e descarte de truncados

In [ ]:
def parece_truncado(ex):
    return not ex["output"].rstrip().endswith((".", "!", "?", ")", ":", '"'))

antes = len(todos_exemplos)

todos_exemplos = [ex for ex in todos_exemplos if not parece_truncado(ex)]

vistos, unicos = set(), []
for ex in todos_exemplos:
    chave = ex["instruction"].strip().lower()
    if chave not in vistos:
        vistos.add(chave)
        unicos.append(ex)
todos_exemplos = unicos

print(f"{antes} -> {len(todos_exemplos)} exemplos (removidos truncados e duplicatas)")


## 7. Normalização do rótulo de desfecho

Padroniza para `DESFECHO: <RÓTULO>` na primeira linha, independente de como o modelo formatou (`Desfecho: X.`, `X — `, `X: `). Isso torna o parsing no LangGraph trivial e determinístico.

A função é idempotente: rodar duas vezes não estraga nada.

In [ ]:
import re

ROTULOS = ("VERIFICAR_EXAMES", "SUGERIR_CONDUTA", "EMITIR_ALERTA")

def normalizar_desfecho(texto):
    return re.sub(
        r'^\s*(?:Desfecho\s*[:\-]?\s*)?(' + "|".join(ROTULOS) + r')\s*[.\-—:–]*\s*',
        r'DESFECHO: \1\n',
        texto
    )

for ex in todos_exemplos:
    if ex.get("categoria") == "cenarios_clinicos":
        ex["output"] = normalizar_desfecho(ex["output"])

# conferir
cenarios = [ex for ex in todos_exemplos if ex.get("categoria") == "cenarios_clinicos"]
com_rotulo = [ex for ex in cenarios if ex["output"].startswith("DESFECHO: ")]
print(f"Cenários com rótulo padronizado: {len(com_rotulo)}/{len(cenarios)}")

from collections import Counter
dist = Counter(ex["output"].split("\n")[0].replace("DESFECHO: ", "") for ex in com_rotulo)
print("\nDistribuição de desfechos:")
for rotulo, n in dist.most_common():
    print(f"  {n:3d}  {rotulo}")

sem_rotulo = [ex for ex in cenarios if not ex["output"].startswith("DESFECHO: ")]
if sem_rotulo:
    print(f"\n⚠️ {len(sem_rotulo)} cenários sem rótulo reconhecível:")
    for ex in sem_rotulo[:3]:
        print("  ", ex["output"][:80])


## 8. Limpeza de meta-vazamento

Remove ocorrências de "fictício/fictícia/sintético" que escaparam para dentro do conteúdo das respostas. As regras específicas vêm antes das genéricas para produzir texto natural ("Identificação do paciente" em vez de "Identificação do ").

Aplicado **só no `output`** — no `input` a marcação "Paciente fictício:" é apropriada e sinaliza que o dado é sintético.

In [ ]:
CORRECOES_META = [
    (r'Identificação\s*\(fictícia,?\s*sem dados reais\)', 'Identificação do paciente'),
    (r'Identificação fictícia do paciente', 'Identificação do paciente'),
    (r'dados antropométricos fictícios', 'dados antropométricos'),
    (r'\bdo paciente fictício\b', 'do paciente'),
    (r'\bpaciente fictício\b', 'paciente'),
    (r'\bfictícios?\b', ''),
    (r'\bfictícias?\b', ''),
    (r'\bsintéticos?\b', ''),
    (r'\(\s*\)', ''),
    (r'\s+([,.;:])', r'\1'),
    (r'[ \t]{2,}', ' '),
]

for ex in todos_exemplos:
    for padrao, subst in CORRECOES_META:
        ex["output"] = re.sub(padrao, subst, ex["output"], flags=re.IGNORECASE)
    ex["output"] = ex["output"].strip()

TERMOS_META = ["fictício", "fictícia", "sem dados reais", "sintético"]
restantes = [ex for ex in todos_exemplos if any(t in ex["output"].lower() for t in TERMOS_META)]
print(f"Meta-vazamentos restantes no output: {len(restantes)}")
for ex in restantes[:3]:
    print("  -", ex["output"][:110])


## 9. Anonimização — SEMPRE POR ÚLTIMO

Versão reescrita. A v2 tinha três regex de iniciais em cascata que deixavam resíduo (`[INICIAIS_REMOVIDAS]S.` quando o nome tinha quatro iniciais). Aqui, um único padrão `{2,}` captura qualquer quantidade numa passada.

**Sobre o regex de inicial única** (`\b[A-Z]\.\b`): não use. Ele destrói texto médico legítimo — "E. coli" vira "[INICIAL_REMOVIDA] coli", "Vitamina D." perde o D. O padrão `{2,}` exige pelo menos duas iniciais consecutivas, que é o formato real de identificação de paciente.

Limitação conhecida: sequências como "Escore A. B. C." seriam capturadas por engano. Raro em texto clínico, mas vale conferir na verificação final.

In [ ]:
def anonimizar(texto: str) -> str:
    # identificadores numéricos
    texto = re.sub(r'\d{3}\.\d{3}\.\d{3}-\d{2}', '[CPF_REMOVIDO]', texto)          # CPF
    texto = re.sub(r'\b\d{2}/\d{2}/\d{4}\b', '[DATA_REMOVIDA]', texto)              # data
    texto = re.sub(r'\b\d{15}\b', '[CNS_REMOVIDO]', texto)                          # cartão nacional de saúde

    # iniciais: 2 ou mais em sequência (R.S. / R.M.T.S.) — uma passada só
    texto = re.sub(r'\b(?:[A-ZÀ-Ú]\.\s*){2,}', '[INICIAIS_REMOVIDAS] ', texto)

    # nomes próprios após marcador de pessoa
    texto = re.sub(
        r'\b(paciente|Paciente|Sr\.|Sra\.)\s+[A-ZÀ-Ú][a-zà-ú]+(?:\s+[A-ZÀ-Ú][a-zà-ú]+)+',
        r'\1 [NOME_REMOVIDO]', texto
    )

    # cosmética: pontuação e espaços após substituição
    texto = re.sub(r'\[(INICIAIS_REMOVIDAS|NOME_REMOVIDO)\]\s+([,.;:)])', r'[\1]\2', texto)
    texto = re.sub(r'[ \t]{2,}', ' ', texto)
    return texto.strip()

for ex in todos_exemplos:
    for campo in ("instruction", "input", "output"):
        ex[campo] = anonimizar(ex[campo])

print("Anonimização aplicada.")


## 10. Verificação final

Roda todas as checagens de uma vez. Use estes números no relatório técnico.

A lista de termos de guardrail foi ampliada: a v2 usava só cinco frases e marcava como "sem guardrail" respostas que tinham a ressalva em outras palavras ("conduta não deve ser iniciada sem confirmação"), subestimando a cobertura real.

In [ ]:
TERMOS_GUARDRAIL = [
    "validação do médico", "validação humana", "médico responsável",
    "requer validação", "apoio à decisão", "decisão clínica explícita",
    "não emite conduta", "não deve ser iniciada sem", "sem confirmação",
    "conduta definitiva", "validação prévia", "confirmação médica",
]

def tem_guardrail(ex):
    return any(t in ex["output"].lower() for t in TERMOS_GUARDRAIL)

def tem_fonte(ex):
    low = ex["output"].lower()
    return any(p in low for p in ("prot-", "laudo-", "rx-", "farm-", "anat-", "coleta-"))

n = len(todos_exemplos)
print("=" * 55)
print(f"TOTAL: {n} exemplos")
print("=" * 55)

print("\nDistribuição por categoria:")
for cat, q in Counter(ex["categoria"] for ex in todos_exemplos).most_common():
    print(f"  {q:3d}  {cat}")

n_guard = sum(1 for ex in todos_exemplos if tem_guardrail(ex))
n_fonte = sum(1 for ex in todos_exemplos if tem_fonte(ex))
print(f"\nCom guardrail:        {n_guard:3d}/{n}  ({100*n_guard//max(n,1)}%)")
print(f"Com citação de fonte: {n_fonte:3d}/{n}  ({100*n_fonte//max(n,1)}%)")

print("\nGuardrail por categoria (esperado: alto em conduta, baixo em informativo):")
for cat in CATEGORIAS:
    da_cat = [ex for ex in todos_exemplos if ex["categoria"] == cat]
    if da_cat:
        g = sum(1 for ex in da_cat if tem_guardrail(ex))
        print(f"  {cat:20s} {g:3d}/{len(da_cat):3d}  ({100*g//len(da_cat)}%)")

print("\n--- Checagens de integridade ---")
checagens = {
    "Truncados": [ex for ex in todos_exemplos if parece_truncado(ex)],
    "Duplicatas": [t for t, q in Counter(ex["instruction"] for ex in todos_exemplos).items() if q > 1],
    "Resíduo de iniciais": [ex for ex in todos_exemplos
        if re.search(r'\[INICIAIS_REMOVIDAS\]\s*[A-ZÀ-Ú]\.', " ".join(str(ex.get(c,"")) for c in ("instruction","input","output")))],
    "Iniciais não anonimizadas": [ex for ex in todos_exemplos
        if re.search(r'\b[A-ZÀ-Ú]\.\s?[A-ZÀ-Ú]\.', " ".join(str(ex.get(c,"")) for c in ("instruction","input","output")))],
    "Meta-vazamento": [ex for ex in todos_exemplos if any(t in ex["output"].lower() for t in TERMOS_META)],
    "Espaços duplos": [ex for ex in todos_exemplos if "  " in ex["output"]],
}
for nome, lista in checagens.items():
    status = "OK" if not lista else f"⚠️  {len(lista)}"
    print(f"  {nome:28s} {status}")

problemas = sum(len(v) for v in checagens.values())
print("\n" + ("✅ Dataset íntegro — pode salvar." if problemas == 0
              else f"⚠️  {problemas} item(ns) para revisar antes de salvar."))


## 11. Curadoria manual

A verificação automática mede forma, não conteúdo. Leia uma amostra de cada categoria — LLMs geram exemplos plausíveis que podem ser repetitivos, genéricos ou clinicamente estranhos, e nenhum regex detecta isso.

In [ ]:
import random

for cat in CATEGORIAS:
    da_cat = [ex for ex in todos_exemplos if ex["categoria"] == cat]
    if not da_cat:
        continue
    ex = random.choice(da_cat)
    print(f"\n{'='*20} {cat} {'='*20}")
    print("INSTRUÇÃO:", ex["instruction"])
    if ex.get("input"):
        print("INPUT:", ex["input"])
    print("RESPOSTA:", ex["output"])
    print(f"[guardrail: {tem_guardrail(ex)} | fonte: {tem_fonte(ex)}]")


In [ ]:
# Remoção manual — liste, escolha os índices ruins, remova
# for i, ex in enumerate(todos_exemplos):
#     print(i, "|", ex["categoria"][:12], "|", ex["instruction"][:70])

# indices_remover = []
# todos_exemplos = [ex for i, ex in enumerate(todos_exemplos) if i not in indices_remover]
# print(f"Restaram {len(todos_exemplos)} exemplos")


## 12. Salvar e baixar

O campo `categoria` fica no arquivo — serve para a análise de distribuição no relatório e para montar um conjunto de avaliação separado por tipo de tarefa. O notebook de treino ignora esse campo.

In [ ]:
NOME_ARQUIVO = "dataset_medico.jsonl"

with open(NOME_ARQUIVO, "w", encoding="utf-8") as f:
    for ex in todos_exemplos:
        registro = {
            "instruction": ex["instruction"],
            "input": ex.get("input", ""),
            "output": ex["output"],
            "categoria": ex.get("categoria", ""),
        }
        f.write(_json.dumps(registro, ensure_ascii=False) + "\n")

print(f"Salvo: {NOME_ARQUIVO} ({len(todos_exemplos)} exemplos)")

# releitura de sanidade — garante que o arquivo é JSONL válido
with open(NOME_ARQUIVO, "r", encoding="utf-8") as f:
    relidos = [_json.loads(l) for l in f]
print(f"Releitura OK: {len(relidos)} registros válidos")


In [ ]:
from google.colab import files
files.download(NOME_ARQUIVO)


## Para o relatório técnico

Copie os números da seção 10. Pontos que valem ser registrados:

**Processo**
- Dados gerados por LLM (Claude Sonnet) a partir de prompt com regras explícitas, em lotes de 5 para evitar truncamento
- Pipeline de limpeza automatizado: deduplicação, normalização de rótulos, remoção de meta-vazamento
- Anonimização em duas camadas (restrição no prompt + regex de pós-processamento)
- Curadoria humana sobre amostra de cada categoria

**Limitações a declarar** — declarar limitação vale mais que omiti-la:
- Protocolos, códigos (PROT-0XX) e doses são inventados; a exatidão regulatória (Portaria 344, prazos, posologia) **não foi validada por profissional de saúde**
- Anonimização por regex tem limitações conhecidas em casos de borda; ferramentas dedicadas (ex.: Microsoft Presidio) seriam o caminho em produção
- O volume (~95 exemplos) dimensiona uma demonstração, não um modelo de produção
- Com LoRA neste volume, o modelo aprende **formato e comportamento** (citar fonte, pedir validação, emitir rótulo de desfecho), não conhecimento clínico — o conhecimento factual vem do RAG

**⚠️ Deixe explícito no README:** protocolos e doses são sintéticos e não devem ser usados clinicamente.

## Célula para o notebook de treino

Substitui a lista `raw_examples` hardcoded.

In [ ]:
import json

with open("dataset_medico.jsonl", "r", encoding="utf-8") as f:
    raw_examples = [json.loads(linha) for linha in f]

print(f"Total de exemplos carregados: {len(raw_examples)}")


## Bônus: extração do desfecho no LangGraph

Com o rótulo padronizado na seção 7, o roteador do seu grafo fica assim:

```python
import re

def extrair_desfecho(resposta: str) -> str:
    m = re.match(r'^DESFECHO:\s*(\w+)', resposta.strip())
    return m.group(1) if m else "SUGERIR_CONDUTA"  # fallback conservador

def rotear(state: EstadoClinico) -> str:
    desfecho = extrair_desfecho(state["resposta_llm"])
    return {
        "VERIFICAR_EXAMES": "verificar_exames",
        "EMITIR_ALERTA": "emitir_alerta",
        "SUGERIR_CONDUTA": "sugerir_conduta",
    }.get(desfecho, "sugerir_conduta")

grafo.add_conditional_edges("assistente", rotear, {
    "verificar_exames": "no_verificar_exames",
    "emitir_alerta": "no_emitir_alerta",
    "sugerir_conduta": "no_sugerir_conduta",
})
```

O fallback aponta para `sugerir_conduta` — que sempre carrega o guardrail de validação humana. Se o parsing falhar, o sistema degrada para o caminho que exige revisão médica, nunca para um que a dispense.